# ***Data Understanding and EDA***

**Project goal**

The project predicts the Remaining Useful Life (RUL) of an aircraft engine: the number of operating cycles left before the end of its simulated life. The prediction will later use sensor readings, operating settings, and cycle history.

**Purpose of this phase:**

Before building a model, we need to understand the dataset, confirm its quality, create the RUL target, and identify useful or unusual features. This reduces errors and helps us choose appropriate preprocessing steps.

**Step 1 — Environment and data setup**

In [ ]:

#to create and work with folder and file paths
from pathlib import Path
#to download data from google drive
import gdown
# /content is Colab's temporary working storage
DATA_DIR = Path("/content/data")
# parents=True creates missing parent folders
# exist_ok=True prevents an error if the folder already exists
DATA_DIR.mkdir(parents=True, exist_ok=True)

FOLDER_URL = "https://drive.google.com/drive/folders/1W6p4ibvdL8wLCLKMm53ImTYZBUgW7Kxc?usp=drive_link"


# Download all files from the shared Google Drive folder
gdown.download_folder(
    url=FOLDER_URL,
    output=str(DATA_DIR),
    quiet=False,
    use_cookies=False,
)



# List the files that the project requires
expected_files = [
    "train_FD001.txt",
    "test_FD001.txt",
    "RUL_FD001.txt",
    "readme.txt",
]
#Check whether each required file exists inside DATA_DIR.
missing_files = [
    filename
    for filename in expected_files
    if not (DATA_DIR / filename).exists()
]
# If one or more files are missing, stop the program and display their names
if missing_files:
    raise FileNotFoundError(f"Missing files: {missing_files}")

print("Setup complete. All FD001 files are available.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


**Step 2 — Define the dataset structure and load the files**

The raw C-MAPSS text files do not contain column headers, so meaningful names must be assigned before the data can be interpreted the names are assined according to the reedme file of the data.

**What we discover:** Each engine is a time series (each row is a cycle of someengine )of sensor readings ordered by cycle. Training data continues until failure, while test data stops before failure.

In [ ]:
id_columns=['engine_id','cycle']
setting_colums=['setting_1','setting_2','setting_3']
sensor_columns=[f'sensor{i}'for i in range(1,22)]
columns=id_columns+setting_colums+sensor_columns
print('number of columns',len(columns))
print('columns names',columns)

In [ ]:
train_df=pd.read_csv(DATA_DIR / "train_FD001.txt",sep='\s+',header=None,names=columns)
test_df=pd.read_csv(DATA_DIR / "test_FD001.txt",sep='\s+',header=None,names=columns)
test_rul_df=pd.read_csv(DATA_DIR / "RUL_FD001.txt",sep='\s+',header=None,names=['true_rul'])




In [ ]:
print(train_df.shape)
train_df.head()

In [ ]:
print(test_df.shape)
test_df.head()

In [ ]:
print(test_rul_df.shape)
test_rul_df.head()

**Step 3 — Validate data quality and sequence structure**

 Missing values, duplicated engine-cycle pairs, or incorrectly ordered cycles can produce unreliable analysis and may create time-series leakage later.

In [ ]:

def check_dataset(df,dataset_name):

  missing_values=df.isna().sum().sum()
  duplicate_cycles=df.duplicated(subset=['engine_id','cycle']).sum()
  all_columns_numeric = all(pd.api.types.is_numeric_dtype(df[column])for column in df.columns)
  positive_id=(df['engine_id']>0).all()
  positive_cycles=(df['cycle']>0).all()
  #If cycles are incorrectly ordered previous cycle may actually refer to a future measurement causing data leakage.
  df_cycle_order=(df.groupby('engine_id')['cycle'].apply(lambda c:c.is_monotonic_increasing).all())
  consecutive_cycles=(df.groupby('engine_id')['cycle'].diff().dropna().eq(1).all())
  engine_num=df['engine_id'].nunique()


  print(f"--- {dataset_name} ---")
  print("Missing values:", missing_values)
  print("Duplicate engine-cycle rows:", duplicate_cycles)
  print("All columns numeric:", all_columns_numeric)
  print("Engine IDs positive:", positive_id)
  print("Cycles positive:", positive_cycles)
  print("Cycles ordered:",  df_cycle_order)
  print("Cycles consecutive:", consecutive_cycles)
  print("Number of engines:", engine_num)
  print()








these checks confirm whether the training and testing datasets follow the expected FD001 structure before RUL calculation or modeling.

In [ ]:
check_dataset(train_df, 'Training dataset')
check_dataset(test_df, 'Testing dataset')

print('--- RUL dataset ---')
print('Shape:', test_rul_df.shape)
print('Missing values:', test_rul_df.isna().sum().sum())
print("Negative RUL values:", (test_rul_df["true_rul"] < 0).sum())

**Step 4 - Explore engine lifetimes**

Engines do not all fail after the same number of cycles. Understanding lifetime variation shows the range of degradation histories the model will need to learn.

In [ ]:
engine_life= train_df.groupby('engine_id')['cycle'].max()
engine_life.describe()

as the graph shows Engine lifetimes vary, so the dataset contains both shorter- and longer-lived engines rather than one fixed failure time.

In [ ]:
plt.figure(figsize=(8,4))

sns.histplot(engine_life, bins=15, kde=True)

plt.title("Distribution of Engine Lifetimes")
plt.xlabel("Maximum Number of Cycles")
plt.ylabel("Number of Engines")
plt.show()

In [ ]:
constant_sensors =[]

for s in sensor_columns:
  if train_df[s].nunique()==1:
    constant_sensors.append(s)

print('constant_sensors',constant_sensors)

clean_train=train_df.drop(columns=constant_sensors).copy()
clean_test=test_df.drop(columns=constant_sensors).copy()






**step5- Detect constant sensors and inspect sensor behavior**

A sensor that never changes cannot help distinguish healthy and degraded engine states. Removing constant sensors reduces unnecessary features

what we discover:Constant sensors carry no predictive variation and are removed consistently from both datasets

In [ ]:
engine1=clean_train[clean_train['engine_id']==1]

for s in sensor_columns:
  if s not in constant_sensors:
    plt.figure(figsize=(4,2))
    plt.plot(engine1['cycle'],engine1[s])
    plt.title(f"Sensor {s} Over Time for Engine 1")
    plt.xlabel("Cycle")
    plt.ylabel(f'reading of Sensor {s} ')
    plt.show()


**Step 6 — Calculate and inspect Remaining Useful Life**
RUL is the numerical target that the machine-learning model will predict.

RUL decreases as an engine advances through its cycles and reaches 0 at its final training record.

In [ ]:
max_cycle= clean_train.groupby('engine_id')['cycle'].transform("max")
clean_train['RUL']=max_cycle-clean_train['cycle']
print( 'number of missing values ',clean_train['RUL'].isna().sum())
clean_train['RUL'].describe()


In [ ]:
plt.figure(figsize=(8, 4))

sns.histplot(
    clean_train["RUL"],
    bins=45
)

plt.title("Distribution of Remaining Useful Life")
plt.xlabel("RUL in Cycles")
plt.ylabel("Number of Records")

plt.show()

**Step 7 — Examine operating settings**

Operating settings should be checked for variation just like sensors. A constant setting cannot help the model explain differences between records.



In [ ]:
print(clean_train[setting_colums].nunique())

setting_3 has only one unique value in FD001, so it is removed consistently from training and testing data.

In [ ]:
#print(clean_train[setting_colums].nunique())
clean_train=clean_train.drop(columns='setting_3')
clean_test=clean_test.drop(columns='setting_3')


In [ ]:
available_sensors=[c for c in clean_train.columns if c.startswith("sensor") ]
sensor_correlations = clean_train[available_sensors].corrwith(clean_train["RUL"])
strong_correlations=sensor_correlations .abs().sort_values(ascending=False)
print(strong_correlations)


**Step 8 — Measure sensor correlation with RUL**

Why we do it: Correlation provides a simple first view of which sensors have the strongest linear relationship with the prediction target.

The strongest-correlated sensors may be useful, but correlation alone does not prove model importance and does not capture nonlinear relationships.

In [ ]:
top_correlations = (
    strong_correlations
    .head(10)
    .sort_values()
)

top_correlations.plot(
    kind="barh",
    figsize=(6,3)
)

plt.title("Top Sensors Correlated with RUL")
plt.xlabel("Absolute Correlation")
plt.ylabel("Sensor")

plt.show()

**Step 9 — Identify potential outliers with the IQR rule**
Why we do it: Unusually high or low readings should be identified before preprocessing. In predictive maintenance, however, an unusual reading may represent real degradation rather than an error.

In [ ]:
Q1=clean_train[available_sensors].quantile(0.25)
Q3=clean_train[available_sensors].quantile(0.75)
IQR=Q3-Q1
lower_bound=Q1-1.5*IQR
upper_bound=Q3+1.5*IQR

outlier_mask=((clean_train[available_sensors]<lower_bound)| (clean_train[available_sensors]>upper_bound))

outlier_count=outlier_mask.sum().sort_values(ascending=False)



In [ ]:
outlier_percenrtage=(outlier_count/len(clean_train))*100
top_outliers = (outlier_percenrtage.head(5).sort_values())


top_outliers.plot(
    kind="barh",
    figsize=(6, 3)
)

plt.title("Sensors with the Most Potential Outliers")
plt.xlabel("Percentage of Records")
plt.ylabel("Sensor")

plt.show()

# Step 10 — Feature Engineering: Understanding sensor changes

**Purpose:** Test whether short-term sensor behavior adds useful information beyond the 18 original model inputs. We selected `sensor2`, `sensor3`, and `sensor7` for this experiment; this is not a claim that they are the only relevant sensors.

**What this cell does:** `selected_sensors` identifies the three signals. `.copy()` protects the cleaned datasets produced during EDA: the new columns are added to separate dataframes, not to your teammate's `clean_train` and `clean_test`.

**Expected result:** Two copies of the cleaned data, ready for feature creation. No model is trained here.

In [ ]:

# Select sensors for initial feature engineering
selected_sensors = ["sensor2", "sensor3", "sensor7"]

# Work on copies to preserve the processed datasets
train_features = clean_train.copy()
test_features = clean_test.copy()

print("Original training shape:", clean_train.shape)
print("Original testing shape:", clean_test.shape)


### Step 10.1 — Create nine time-based features

**Why:** A single reading describes the current sensor value, but it does not directly describe its recent direction or variation. For each of three sensors, we create:
- **diff:** Measures the change between two consecutive cycles.
- **mean5:** Calculates the average of the last five readings to show the trend.
- **std5:** Measures how much the readings vary over the last five cycles.

This adds 9 features to the original 18, giving us 27 features to test.

In [ ]:

def build_features(df):
    df = df.sort_values(
        ["engine_id", "cycle"]
    ).copy()

    for sensor in selected_sensors:
        group = df.groupby("engine_id")[sensor]

        # Difference from previous cycle
        df[f"{sensor}_diff"] = (
            group.diff().fillna(0)
        )

        # Average of the last 5 readings
        df[f"{sensor}_mean5"] = (
            group.transform(
                lambda x: x.rolling(
                    window=5,
                    min_periods=1
                ).mean()
            )
        )

        # Standard deviation of last 5 readings
        df[f"{sensor}_std5"] = (
            group.transform(
                lambda x: x.rolling(
                    window=5,
                    min_periods=1
                ).std()
            ).fillna(0)
        )

    return df

train_features = build_features(clean_train)
test_features = build_features(clean_test)

print("New training shape:", train_features.shape)
print("New testing shape:", test_features.shape)


### Step 10.2 — Visualize what the new features mean

**Why:** Numbers alone make it hard to see what `diff`, `mean5`, and `std5` represent. We plot one example engine (`engine_id = 1`) and one selected sensor (`sensor2`) to illustrate the transformations.

**Reading the plots:** The top plot compares raw sensor readings with the five-cycle rolling mean. The bottom plot shows cycle-to-cycle changes and local standard deviation. These curves are descriptive, not proof of noise removal or fault detection. The actual prediction benefit is tested in Step 11.

**Expected result:** Two compact plots, rather than many repeated figures.

In [ ]:
import matplotlib.pyplot as plt
engine_example = train_features.loc[train_features["engine_id"] == 1]
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(engine_example["cycle"], engine_example["sensor2"], alpha=0.6, label="Raw sensor2")
axes[0].plot(engine_example["cycle"], engine_example["sensor2_mean5"], label="Rolling mean (5)")
axes[0].set_ylabel("Sensor reading")
axes[0].legend()
axes[1].plot(engine_example["cycle"], engine_example["sensor2_diff"], label="Cycle difference")
axes[1].plot(engine_example["cycle"], engine_example["sensor2_std5"], label="Rolling std (5)")
axes[1].set_xlabel("Cycle")
axes[1].set_ylabel("Derived values")
axes[1].legend()
plt.tight_layout()
plt.show()


### Step 10.3 — Validate the engineered dataset

**Why:** Before evaluating a model, confirm that feature generation did not lose observations, introduce missing values, or accidentally drop the original model inputs.

**Checks:** `original_features` excludes the engine identifier and RUL target, leaving the original 18 inputs (including `cycle`); `engineered_columns` lists the nine new features. `assert` stops execution if a requirement fails. We check train/test row counts, missing values, and that all model input columns exist in both datasets.

**Expected result:** A short confirmation: `Original: 18 features | Engineered: 27 features`. This checks structure, not predictive quality.

In [ ]:
engineered_columns = [f"{sensor}_{suffix}" for sensor in selected_sensors for suffix in ("diff", "mean5", "std5")]
# `cycle` is retained as an input because it was used in our original Random Forest experiments.
original_features = [c for c in clean_train.columns if c not in ("engine_id", "RUL")]
assert len(original_features) == 18
assert len(engineered_columns) == 9
assert len(train_features) == len(clean_train) and len(test_features) == len(clean_test)
assert train_features[engineered_columns].isna().sum().sum() == 0
assert test_features[engineered_columns].isna().sum().sum() == 0
assert all(c in test_features.columns for c in original_features + engineered_columns)
print(f"Original: {len(original_features)} features | Engineered: {len(original_features)+len(engineered_columns)} features")


# Step 11 — Compare three feature approaches on unseen engines

**Research question:** Do the nine engineered features or PCA improve RUL prediction enough to justify additional complexity?

We evaluate the **same held-out engines**, the **same RUL target**, and the **same Random Forest settings** for all three approaches:

1. **Original (18):** the cleaned sensor/setting/cycle inputs.
2. **Engineered (27):** the original 18 plus the nine time-based features.
3. **PCA (12 components):** scale the original 18 inputs, then transform them into **12 principal components**. PCA is a dimensionality-reduction method, not an additional nine sensor features.

**Why engine-level splitting:** Each engine has many cycles. Splitting individual rows could place cycles from the same engine in both training and validation. `GroupShuffleSplit` keeps each engine wholly in one side.

**Metrics:** Lower **MAE** and **RMSE** are better; higher **R²** is better. MAE is the average absolute RUL prediction error in cycles.

**Scope:** This is a controlled feature comparison, **not** final model training, NASA test evaluation, or hyperparameter tuning.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(clean_train, groups=clean_train["engine_id"]))
assert set(clean_train.iloc[train_idx]["engine_id"]).isdisjoint(set(clean_train.iloc[val_idx]["engine_id"]))
y_train = clean_train["RUL"].iloc[train_idx]
y_val = clean_train["RUL"].iloc[val_idx]
print("Training engines:", clean_train.iloc[train_idx]["engine_id"].nunique())
print("Validation engines:", clean_train.iloc[val_idx]["engine_id"].nunique())


In [ ]:
def evaluate_features(name, x_train, x_val):
    model = RandomForestRegressor(n_estimators=100, max_depth=15,
                                  min_samples_leaf=3, random_state=42, n_jobs=-1)
    model.fit(x_train, y_train)
    pred = model.predict(x_val)
    return {"Approach": name,
            "Features": x_train.shape[1],
            "MAE": mean_absolute_error(y_val, pred),
            "RMSE": np.sqrt(mean_squared_error(y_val, pred)),
            "R2": r2_score(y_val, pred)}

results = []
results.append(evaluate_features("Original", clean_train.iloc[train_idx][original_features],
                                 clean_train.iloc[val_idx][original_features]))
all_features = original_features + engineered_columns
results.append(evaluate_features("Engineered", train_features.iloc[train_idx][all_features],
                                 train_features.iloc[val_idx][all_features]))

scaler = StandardScaler()
x_pca_train_scaled = scaler.fit_transform(clean_train.iloc[train_idx][original_features])
x_pca_val_scaled = scaler.transform(clean_train.iloc[val_idx][original_features])
pca = PCA(n_components=12)
x_pca_train = pca.fit_transform(x_pca_train_scaled)
x_pca_val = pca.transform(x_pca_val_scaled)
results.append(evaluate_features("PCA", x_pca_train, x_pca_val))
comparison = pd.DataFrame(results)
display(comparison.round(3))


### Step 11.1 — Visualize the single-split comparison

**Why:** The table in Step 11 shows exact MAE/RMSE/R² values; a single bar chart makes the MAE differences easier to see.

**How to read:** Shorter bars indicate lower validation MAE. A small difference between bars does not prove that one approach will consistently win on other engines. We therefore also run grouped cross-validation below.

In [ ]:
ax = comparison.plot.bar(x="Approach", y="MAE", legend=False, figsize=(8, 4), rot=0)
ax.set_ylabel("Validation MAE (cycles)")
ax.set_title("Feature approaches — same engine-level validation split")
plt.tight_layout()
plt.show()


### Step 11.2 — Group Cross-Validation (3 folds)

**Why:** A single hold-out result depends on which engines happen to be selected. `GroupKFold(n_splits=3)` evaluates each representation on **three different sets of unseen engines**, keeping all cycles of any engine within one fold.

**Fair comparison:** The same folds, RUL target, **100 trees**, `max_depth=15`, and `min_samples_leaf=3` are used for all three approaches. In our earlier tree-count tuning, 100 trees performed slightly better than 50; we keep 100 fixed here rather than tuning it again. These are **comparison settings**, not the separate final model's hyperparameters.

**Preventing leakage:** For PCA, `StandardScaler` and `PCA(n_components=12)` are placed inside a `Pipeline`. Each fold fits both transformations **only on that fold's training engines**, then applies them to its validation engines.

**What the code reports:** For each approach, the mean MAE, variation in MAE across folds (`MAE Std`), mean RMSE, and mean R². Smaller mean MAE is preferable, but the size and consistency of the differences matter. This cross-validation does **not** use NASA's held-out test engines.

In [ ]:
cv = GroupKFold(n_splits=3)
groups = clean_train["engine_id"]
y = clean_train["RUL"]
feature_sets = {
    "Original (18)": clean_train[original_features],
    "Engineered (27)": train_features[original_features + engineered_columns],
    "PCA (12)": clean_train[original_features],
}

cv_rows = []
for approach, X in feature_sets.items():
    fold_mae, fold_rmse, fold_r2 = [], [], []
    for fold_train, fold_val in cv.split(X, y, groups=groups):
        rf = RandomForestRegressor(
            n_estimators=100, max_depth=15, min_samples_leaf=3,
            random_state=42, n_jobs=-1
        )
        model = (Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=12)),
            ("rf", rf)
        ]) if approach == "PCA (12)" else rf)

        model.fit(X.iloc[fold_train], y.iloc[fold_train])
        pred = model.predict(X.iloc[fold_val])
        fold_mae.append(mean_absolute_error(y.iloc[fold_val], pred))
        fold_rmse.append(np.sqrt(mean_squared_error(y.iloc[fold_val], pred)))
        fold_r2.append(r2_score(y.iloc[fold_val], pred))

    cv_rows.append({
        "Approach": approach,
        "Mean MAE": np.mean(fold_mae),
        "MAE Std": np.std(fold_mae),
        "Mean RMSE": np.mean(fold_rmse),
        "Mean R2": np.mean(fold_r2)
    })

cv_comparison = pd.DataFrame(cv_rows).sort_values("Mean MAE")
display(cv_comparison.round(3))


### Interpretation — Feature Selection

We compared three approaches: Original (18), Engineered (27), and PCA (12).

The 9 engineered features gave only a small improvement. PCA performed slightly better in single validation, but the original features performed better in Cross-Validation.

We chose the **18 original features** because they gave good results, were easier to understand, and kept the original sensor readings without extra transformations.

We saved the cleaned and engineered training and testing datasets as CSV files.

This allows our teammates to use the same prepared data for Random Forest and Anomaly Detection without repeating the preprocessing steps.

In [ ]:
EXPORT_DIR = DATA_DIR / "processed"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
clean_train.to_csv(EXPORT_DIR / "train_original.csv", index=False)
clean_test.to_csv(EXPORT_DIR / "test_original.csv", index=False)
train_features.to_csv(EXPORT_DIR / "train_engineered.csv", index=False)
test_features.to_csv(EXPORT_DIR / "test_engineered.csv", index=False)
test_rul_df.to_csv(EXPORT_DIR / "test_rul.csv", index=False)
print("Exported datasets to:", EXPORT_DIR)
